<a href="https://colab.research.google.com/github/VinitRaj31/Drug_Discovery_using_ML/blob/main/Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install chembl_webresource_client
! pip install rdkit

In [ ]:
#ccess to mathematical functions like sqrt, log
import math

'''
this module offers an object-oriented way to interact with the file system,
making path manipulations cleaner and more intuitive
'''
from pathlib import Path

#Provides tools to work with ZIP archives (compress and decompress files).
from zipfile import ZipFile

#Creates and manages temporary directories, useful for intermediate files that don't need to persist.
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

#This module from RDKit integrates chemical functionalities with Pandas DataFrames
from rdkit.Chem import PandasTools

#This client is used to access the ChEMBL database
from chembl_webresource_client.new_client import new_client

#extensible progress bars for loops
from tqdm.auto import tqdm

In [ ]:
HERE = Path.cwd()
# This is a convenient way to define a path to a 'data' folder relative to the current working directory.
DATA = HERE / "data"

In [ ]:
#allows you to search for and retrieve information about biological targets
targets_api = new_client.target
compounds_api = new_client.molecule
bioactivities_api = new_client.activity

In [ ]:
uniprot_id = "P00533"

In [ ]:
# Get target information from ChEMBL but restrict it to specified values only
targets = targets_api.get(target_components__accession=uniprot_id).only(
    "target_chembl_id", "organism", "pref_name", "target_type"
)
print(f'The type of the targets is "{type(targets)}"')

In [ ]:
#retriving all the targets which have p00533 as one of their component
targets = pd.DataFrame.from_records(targets)
targets

In [ ]:
target = targets.iloc[0]
target

In [ ]:
chembl_id = target.target_chembl_id
print(f"The target ChEMBL ID is {chembl_id}")

In [ ]:
bioactivities = bioactivities_api.filter(
    target_chembl_id=chembl_id, type="IC50", relation="=", assay_type="B"
).only(
    "activity_id",
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "molecule_chembl_id",
    "type",
    "standard_units",
    "relation",
    "standard_value",
    "target_chembl_id",
    "target_organism",
)

print(f"Length and type of bioactivities object: {len(bioactivities)}, {type(bioactivities)}")

In [ ]:
print(f"Length and type of first element: {len(bioactivities[0])}, {type(bioactivities[0])}")
bioactivities[0]

In [ ]:
#Downloading Data
bioactivities_df = pd.DataFrame.from_dict(bioactivities)
print(f"DataFrame shape: {bioactivities_df.shape}")
bioactivities_df.head()

In [ ]:
bioactivities_df["units"].unique()

In [ ]:
bioactivities_df.drop(["units", "value"], axis=1, inplace=True)
bioactivities_df.head()

Data Processing and Filteration

Delete entries with missing and duplicate values.

Reset index number and rename columns

In [ ]:
bioactivities_df.dtypes
bioactivities_df = bioactivities_df.astype({"standard_value": "float64"})
bioactivities_df.dtypes

In [ ]:
#NO of rows and column in the dataframe
bioactivities_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
#cuurent units the dataframe contain
print(f"Units in downloaded data: {bioactivities_df['standard_units'].unique()}")
print(
    f"Number of non-nM entries:\
    {bioactivities_df[bioactivities_df['standard_units'] != 'nM'].shape[0]}"
)

In [ ]:
bioactivities_df = bioactivities_df[bioactivities_df["standard_units"] == "nM"]
print(f"Units after filtering: {bioactivities_df['standard_units'].unique()}")

In [ ]:
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
bioactivities_df.reset_index(drop=True, inplace=True)
bioactivities_df.head()

In [ ]:
bioactivities_df.rename(
    columns={"standard_value": "IC50", "standard_units": "units"}, inplace=True
)
bioactivities_df.head()

In [ ]:
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
bioactivities_df.drop_duplicates("molecule_chembl_id", keep="first", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
bioactivities_df.reset_index(drop=True, inplace=True)
bioactivities_df.head()

In [ ]:

bioactivities_df.rename(
    columns={"standard_value": "IC50", "standard_units": "units"}, inplace=True
)
bioactivities_df.head()

In [ ]:
print(f"DataFrame shape: {bioactivities_df.shape}")

We are going to fetch compound ChEMBL IDs and structures for the compounds linked to our filtered bioactivity data.

In [ ]:
#it selects the molecule_chembl_id and molecule_structures fields for these compounds
compounds_provider = compounds_api.filter(
    molecule_chembl_id__in=list(bioactivities_df["molecule_chembl_id"])
).only("molecule_chembl_id", "molecule_structures")

In [ ]:
compounds = list(tqdm(compounds_provider))

In [ ]:
compounds_df = pd.DataFrame.from_records(
    compounds,
)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
compounds_df.head()

In [ ]:
compounds_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
compounds_df.drop_duplicates("molecule_chembl_id", keep="first", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
compounds_df.iloc[0].molecule_structures.keys()

In [ ]:
canonical_smiles = []

for i, compounds in compounds_df.iterrows():
    try:
        canonical_smiles.append(compounds["molecule_structures"]["canonical_smiles"])
    except KeyError:
        canonical_smiles.append(None)

compounds_df["smiles"] = canonical_smiles
compounds_df.drop("molecule_structures", axis=1, inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
compounds_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

In [ ]:
print(f"Bioactivities filtered: {bioactivities_df.shape[0]}")
bioactivities_df.columns

In [ ]:
print(f"Compounds filtered: {compounds_df.shape[0]}")
compounds_df.columns

**Lets merge both the datasets**

Merge to contain, molecule ID, IC50, Units, and canonical smiles

In [ ]:
# Merge DataFrames
output_df = pd.merge(
    bioactivities_df[["molecule_chembl_id", "IC50", "units"]],
    compounds_df,
    on="molecule_chembl_id",
)

# Reset row indices
output_df.reset_index(drop=True, inplace=True)

print(f"Dataset with {output_df.shape[0]} entries.")

In [ ]:
output_df.dtypes
output_df.head(10)

**Convert IC50 values into pIC50 and put it in an extra column**

In [ ]:
def convert_ic50_to_pic50(IC50_value):
    pIC50_value = 9 - math.log10(IC50_value)
    return pIC50_value

In [ ]:
# Apply conversion to each row of the compounds DataFrame
output_df["pIC50"] = output_df.apply(lambda x: convert_ic50_to_pic50(x.IC50), axis=1)

In [ ]:
output_df.head()

**Distribution Based on IpIC50**

In [ ]:
output_df.hist(column="pIC50")

In [ ]:
# Add molecule column
PandasTools.AddMoleculeColumnToFrame(output_df, smilesCol="smiles")
# Sort molecules by pIC50
output_df.sort_values(by="pIC50", ascending=False, inplace=True)

# Reset index
output_df.reset_index(drop=True, inplace=True)
output_df.drop("smiles", axis=1).head(3)
# Prepare saving the dataset: Drop the ROMol column
output_df = output_df.drop("ROMol", axis=1)
print(f"DataFrame shape: {output_df.shape}")

In [ ]:
output_df.to_csv("/content/sample_data/EGFR_compounds.csv")
output_df.head()